In [2]:
!pip install ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 108.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13


In [3]:
from ultralytics import YOLO

model=YOLO("yolov8n.pt")
print("YOLO est pret ✅")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO est pret ✅


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{\r\n  "username":"hichambakaz2",\r\n  "key":"KGAT_4c2ab09a6e3ae1db50b084a5010744d4"\r\n}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d localbrain/plasticbag

Dataset URL: https://www.kaggle.com/datasets/localbrain/plasticbag
License(s): unknown
100% 12.9M/12.9M [00:00<00:00, 50.8MB/s]



In [ ]:
!unzip plasticbag.zip -d /content/drive/MyDrive/plasticbag_dataset

Archive:  plasticbag.zip
  inflating: /content/drive/MyDrive/plasticbag_dataset/plastic_bag/(10).png  
  inflating: /content/drive/MyDrive/plasticbag_dataset/plastic_bag/(100).png  
  inflating: /content/drive/MyDrive/plasticbag_dataset/plastic_bag/(102).png  
  inflating: /content/drive/MyDrive/plasticbag_dataset/plastic_bag/(103).png  
  inflating: /content/drive/MyDrive/plasticbag_dataset/plastic_bag/(104).png  
  inflating: /content/drive/MyDrive/plasticbag_dataset/plastic_bag/(106).png  
  inflating: /content/drive/MyDrive/plasticbag_dataset/plastic_bag/(11).png  
  inflating: /content/drive/MyDrive/plasticbag_dataset/plastic_bag/(110).png  
  inflating: /content/drive/MyDrive/plasticbag_dataset/plastic_bag/(112).png  
  inflating: /content/drive/MyDrive/plasticbag_dataset/plastic_bag/(113).png  
  inflating: /content/drive/MyDrive/plasticbag_dataset/plastic_bag/(115).png  
  inflating: /content/drive/MyDrive/plasticbag_dataset/plastic_bag/(117).png  
  inflating: /content/drive/M

In [4]:
!pip install roboflow

In [16]:
import os, shutil

# supprimer ancien dataset
if os.path.exists("/content/plastic-bag-1"):
    shutil.rmtree("/content/plastic-bag-1")

print("Ancien dataset supprimé")

Ancien dataset supprimé


In [17]:
from roboflow import Roboflow

rf = Roboflow(api_key="h0gcIDEOsbsQRVnkxO5p")
project = rf.workspace("hichams-workspace-twfan").project("plastic-bag-aqpmj")
dataset = project.version(1).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to plastic-bag-1 in yolov8:: 100%|██████████| 367/367 [00:00<00:00, 4334.52it/s]


In [22]:
import os, shutil, random
from pathlib import Path

base = Path("/content/plastic-bag-1")

train_img = base / "train/images"
train_lbl = base / "train/labels"
valid_img = base / "valid/images"
valid_lbl = base / "valid/labels"

valid_img.mkdir(parents=True, exist_ok=True)
valid_lbl.mkdir(parents=True, exist_ok=True)

images = list(train_img.glob("*"))
random.shuffle(images)

val_count = max(1, int(len(images) * 0.2))

for img in images[:val_count]:
    label = train_lbl / (img.stem + ".txt")
    shutil.move(str(img), str(valid_img / img.name))
    if label.exists():
        shutil.move(str(label), str(valid_lbl / label.name))

print("Train:", len(list(train_img.glob("*"))))
print("Valid:", len(list(valid_img.glob("*"))))

Train: 146
Valid: 36


In [23]:
yaml_text = """
train: /content/plastic-bag-1/train/images
val: /content/plastic-bag-1/valid/images

nc: 1
names: ['plastic_bag']
"""

with open("/content/plastic-bag-1/data.yaml", "w") as f:
    f.write(yaml_text)

In [24]:
model.train(
    data="/content/plastic-bag-1/data.yaml",
    epochs=20,
    imgsz=640
)

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/plastic-bag-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-8, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ec4cf796ab0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [25]:
from google.colab import files
files.download("/content/runs/detect/train-8/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>